In [4]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")


In [5]:
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'


In [6]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "eng_Latn",  # English (baseline)
    "sin": "sin_Sinh",  # Sinhala
    "san": "san_Deva",  # Sanskrit
    "tam": "tam_Taml",  # Tamil
    "hin": "hin_Deva",  # Hindi
    "ben": "ben_Beng",  # Bengali
    "arb": "arb_Arab",  # Arabic (Modern Standard)
    "fra": "fra_Latn",  # French
    "deu": "deu_Latn",  # German
}

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["flores_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    else:
        print("No matching target languages found in this dataset.")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [7]:
import os
import urllib.request
import fasttext

MODEL_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
MODEL_PATH = "models/benchmark/fastText/lid.176.bin"

FASTTEXT_LANG_MAP = {
    "eng_Latn": "en", "sin_Sinh": "si", "san_Deva": "sa", "tam_Taml": "ta",
    "hin_Deva": "hi", "ben_Beng": "bn", "arb_Arab": "ar", "fra_Latn": "fr", "deu_Latn": "de",
}

if not os.path.exists(MODEL_PATH):
    print("Downloading fastText LID-176 model (~126MB)...")
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

print("Loading fastText LID-176 model...")
model = fasttext.load_model(MODEL_PATH)
model_name = "fastText LID-176"
target_labels = sorted(set(FASTTEXT_LANG_MAP.values()))

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].astype(str).str.replace("\n", " ").tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    preds, _ = model.predict(texts, k=1)

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["flores_label"].map(FASTTEXT_LANG_MAP)
    results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]

    evaluate_and_save(results, model_name, dataset_name, target_labels)


Loading fastText LID-176 model...

Loading flores_plus.jsonl...


Loaded 12116 rows across 9 target languages
Evaluating 12116 samples with fastText LID-176...

ZERO-SHOT BENCHMARK RESULTS (fastText LID-176 on flores_plus)
Accuracy:  80.18%
Macro F1:  79.51%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     1.0000    0.5000    0.6667      2024
          bn     1.0000    1.0000    1.0000      1012
          de     0.9797    1.0000    0.9897      1012
          en     0.5641    1.0000    0.7213      1012
          fr     0.9750    1.0000    0.9873      1012
          hi     1.0000    1.0000    1.0000      1012
          sa     0.0000    0.0000    0.0000      1327
          si     0.6647    0.9770    0.7912      2693
          ta     1.0000    1.0000    1.0000      1012

   micro avg     0.8184    0.8018    0.8100     12116
   macro avg     0.7982    0.8308    0.7951     12116
weighted avg     0.7758    0.8018    0.7632     12116



/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipel


Saved predictions to datasets/benchmark_results/fasttext_lid_176_flores_plus.csv


Loading commonlid.jsonl...
Loaded 74052 rows across 9 target languages
Evaluating 74052 samples with fastText LID-176...

ZERO-SHOT BENCHMARK RESULTS (fastText LID-176 on commonlid)
Accuracy:  95.50%
Macro F1:  84.55%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     0.9997    0.9898    0.9947     26152
          bn     0.9984    0.9889    0.9936      1886
          de     0.9717    0.9629    0.9673      7553
          en     0.9851    0.9603    0.9725     27461
          fr     0.9413    0.9480    0.9447      3233
          hi     0.9952    0.9684    0.9816      3666
          sa     0.0000    0.0000    0.0000      1327
          si     0.6646    0.9770    0.7910      2693
          ta     0.9310    1.0000    0.9643        81

   micro avg     0.9701    0.9550    0.9625     74052
   macro avg     0.8319    0.8661    0.8455     74052
weighted avg     0.958

/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/dat